# Train axon-1 or veda-1 on Colab / Kaggle

**Runtime:** Runtime → Change runtime type → **L4 GPU** (or A100). These need Colab
compute units (Pay As You Go, $9.99 for 100 units; an axon-1 run uses roughly 10–20).
A free **T4** also works, but fits only short sessions (4k tokens), so most agent
sessions get skipped. The training cell picks the right settings for whichever GPU you have.

You need trajectories first: either your own (`training.log_trajectories: true` in
AgentCode, saved to `~/.agentcode/trajectories/`) or ones from `finetune/generate_data.py`.
Upload that folder to Google Drive as `MyDrive/agentcode/trajectories/`.

tandav-1 (80B) doesn't fit on Colab; see `finetune/README.md` for multi-GPU training.

In [ ]:
MODEL = "axon-1"          # or "veda-1"
BRANCH = "claude/youthful-hypatia-2enztp"   # use "main" once merged
HF_REPO = ""              # e.g. "your-name/axon-1" to upload the result (optional)

In [ ]:
!git clone -q -b $BRANCH https://github.com/vigp17/AgentCode.git
%cd AgentCode
!pip install -q unsloth trl datasets

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
TRAJ = "/content/drive/MyDrive/agentcode/trajectories"
!ls $TRAJ | head

Keep the 500 highest-reward trajectories for this model and split train/eval by project:

In [ ]:
!python finetune/prepare_data.py --input $TRAJ --model $MODEL --top 500

Optional: to push to the Hugging Face Hub, add an `HF_TOKEN` secret (🔑 in the left
sidebar, with write access) and set `HF_REPO` above.

In [ ]:
import os, torch
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

# Pick settings for this GPU. Agent sessions are long: a 16GB T4 only fits
# ~4k tokens, so most sessions would be skipped. 24GB+ (L4/A100) fits 16k.
gpu = torch.cuda.get_device_name(0)
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if mem_gb < 20:
    flags = "--colab"
    print(f"{gpu} ({mem_gb:.0f}GB): 4k context — long sessions will be skipped. "
          "For better results switch to an L4 or A100 runtime.")
else:
    flags = "--max-seq-length 16384 --no-merged"
    print(f"{gpu} ({mem_gb:.0f}GB): 16k context")
push = f"--push-to-hub {HF_REPO}" if HF_REPO else ""
!python finetune/train.py $MODEL $flags --gguf q4_k_m $push

Save the GGUF (for Ollama on your Mac) and the LoRA adapter to Drive:

In [ ]:
!mkdir -p /content/drive/MyDrive/agentcode/out
!cp -r finetune/out/$MODEL /content/drive/MyDrive/agentcode/out/
!ls -lh finetune/out/$MODEL/gguf

## Run it locally

On your Mac, from the copied `gguf/` folder:

```bash
ollama create axon-1 -f Modelfile
```

Then in your workspace `.env`:

```
AGENTCODE_FT_AXON_1_API_BASE=http://localhost:11434/v1
```

Reload VS Code. axon-1's dot turns green and it answers in the panel.